# Práctico 4 — Análisis PCA
**Curso:** Análisis de Datos  
**Dataset:** Productividad de operarios textiles (`Tema_11_clean.csv`)  
**Objetivo:** Aplicar Análisis de Componentes Principales, construir biplots y evaluar qué características tienen los operarios que cumplen su objetivo de productividad.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid')
print('Paquetes cargados correctamente.')

Paquetes cargados correctamente.


## 1. Carga y exploración inicial del dataset

In [2]:
df = pd.read_csv('Tema_11_clean.csv')
print(f'Dimensiones: {df.shape}')
print('\nPrimeras filas:')
df.head()

Dimensiones: (1013, 15)

Primeras filas:


,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,2015-01-01,Quarter1,sewing,Thursday,8,0.80,26.16,1108.0,7080.0,98.0,0.0,0.0,0.0,59.0,0.940725
1,2015-01-01,Quarter1,finishing,Thursday,1,0.75,3.94,0.0,960.0,0.0,0.0,0.0,0.0,8.0,0.886500
2,2015-01-01,Quarter1,sewing,Thursday,11,0.80,11.41,968.0,3660.0,50.0,0.0,0.0,0.0,30.5,0.800570
3,2015-01-01,Quarter1,sewing,Thursday,12,0.80,11.41,968.0,3660.0,50.0,0.0,0.0,0.0,30.5,0.800570
4,2015-01-01,Quarter1,finishing,Thursday,2,0.75,3.94,0.0,960.0,0.0,0.0,0.0,0.0,8.0,0.755167


In [3]:
print('Tipos de datos:')
print(df.dtypes)
print(f'\nValores nulos por columna:\n{df.isnull().sum()}')
df.describe()

Tipos de datos:
date                         str
quarter                      str
department                   str
day                          str
team                       int64
targeted_productivity    float64
smv                      float64
wip                      float64
over_time                float64
incentive                float64
idle_time                float64
idle_men                 float64
no_of_style_change       float64
no_of_workers            float64
actual_productivity      float64
dtype: object

Valores nulos por columna:
date                     47
quarter                   5
department                0
day                       2
team                      0
targeted_productivity     0
smv                       0
wip                       0
over_time                 0
incentive                 0
idle_time                 0
idle_men                  0
no_of_style_change        0
no_of_workers             0
actual_productivity       0
dtype: int64


,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
count,1013.000000,1013.000000,1013.000000,1013.000000,1013.000000,1013.000000,1013.000000,1013.000000,1013.000000,1013.000000,1013.000000
mean,6.517275,0.730281,15.221678,714.234452,4516.485686,36.474827,0.858835,0.330701,0.144126,35.472853,0.731585
std,3.472214,0.097273,10.762026,1627.009349,3247.633836,132.701747,13.812491,2.976226,0.422861,21.920107,0.175122
min,1.000000,0.350000,2.900000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,0.233705
25%,4.000000,0.700000,3.940000,0.000000,1440.000000,0.000000,0.000000,0.000000,0.000000,9.000000,0.650224
50%,7.000000,0.750000,15.260000,609.000000,3960.000000,0.000000,0.000000,0.000000,0.000000,35.000000,0.770114
75%,9.000000,0.800000,22.940000,1083.000000,6840.000000,50.000000,0.000000,0.000000,0.000000,57.000000,0.850137
max,12.000000,0.800000,54.560000,23122.000000,15120.000000,2880.000000,300.000000,45.000000,2.000000,89.000000,1.000000


## 2. Preprocesamiento — Codificación de variables categóricas

Para no inducir sesgo en el PCA, se aplican distintas estrategias según la naturaleza de cada variable:

| Variable | Tipo | Estrategia |
|---|---|---|
| `department` | Nominal binaria | Codificación binaria (sewing=1, finishing=0) |
| `quarter` | Ordinal | Codificación ordinal (Q1=1…Q5=5) |
| `day` | Cíclica | Codificación cíclica con sin/cos para preservar periodicidad semanal |
| `date` | Redundante | Eliminada (la información temporal ya está en quarter y day) |

In [4]:
# Eliminar columna de fecha (redundante con quarter y day)
df_proc = df.drop(columns=['date']).copy()

# Eliminar filas con valores nulos
df_proc = df_proc.dropna().reset_index(drop=True)
print(f'Registros tras eliminar nulos: {len(df_proc)}')

# --- department: binaria (sewing=1, finishing=0) ---
df_proc['department_enc'] = (df_proc['department'] == 'sewing').astype(int)

# --- quarter: ordinal ---
quarter_map = {'Quarter1': 1, 'Quarter2': 2, 'Quarter3': 3, 'Quarter4': 4, 'Quarter5': 5}
df_proc['quarter_enc'] = df_proc['quarter'].map(quarter_map)

# --- day: codificación cíclica sin/cos ---
# Evita imponer orden artificial (ej. lunes=0 < domingo=6)
day_order = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3,
             'Friday': 4, 'Saturday': 5, 'Sunday': 6}
df_proc['day_num'] = df_proc['day'].map(day_order)
df_proc['day_sin'] = np.sin(2 * np.pi * df_proc['day_num'] / 7)
df_proc['day_cos'] = np.cos(2 * np.pi * df_proc['day_num'] / 7)

# --- Variable objetivo: cumplió o superó productividad objetivo ---
df_proc['met_target'] = (df_proc['actual_productivity'] >= df_proc['targeted_productivity']).astype(int)
print(f"\nCumplieron objetivo: {df_proc['met_target'].sum()} ({df_proc['met_target'].mean()*100:.1f}%)")
print(f"No cumplieron:       {(df_proc['met_target']==0).sum()} ({(1-df_proc['met_target'].mean())*100:.1f}%)")

Registros tras eliminar nulos: 1006

Cumplieron objetivo: 726 (72.2%)
No cumplieron:       280 (27.8%)


In [5]:
# Columnas para PCA y sus etiquetas de visualización
feature_cols = [
    'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive',
    'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers',
    'actual_productivity', 'department_enc', 'quarter_enc',
    'day_sin', 'day_cos', 'team'
]

feature_labels = [
    'Prod. Obj.', 'SMV', 'WIP', 'Horas Extra', 'Incentivo',
    'T. Inactivo', 'Op. Inactivos', 'Cambios Estilo', 'N° Operarios',
    'Prod. Real', 'Depart.', 'Trimestre',
    'Día (sen)', 'Día (cos)', 'Equipo'
]

X = df_proc[feature_cols].values
y = df_proc['met_target'].values
print(f'Matriz de features: {X.shape}')

Matriz de features: (1006, 15)


## 3. Estandarización y PCA

PCA es sensible a la escala de las variables, por lo que se estandariza cada columna a media 0 y desvío estándar 1 antes de aplicar el análisis.

In [6]:
# Estandarización
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA completo
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

var_exp = pca.explained_variance_ratio_
cum_var = np.cumsum(var_exp)
loadings = pca.components_.T  # shape (n_features, n_components)

print('Varianza explicada por componente:')
for i, (v, cv) in enumerate(zip(var_exp[:8], cum_var[:8])):
    print(f'  PC{i+1}: {v*100:5.2f}%   Acumulada: {cv*100:6.2f}%')

Varianza explicada por componente:
  PC1: 24.41%   Acumulada:  24.41%
  PC2: 11.75%   Acumulada:  36.17%
  PC3:  9.10%   Acumulada:  45.26%
  PC4:  8.04%   Acumulada:  53.30%
  PC5:  6.92%   Acumulada:  60.22%
  PC6:  6.72%   Acumulada:  66.94%
  PC7:  6.60%   Acumulada:  73.55%
  PC8:  5.80%   Acumulada:  79.35%


## 4. Scree Plot y varianza acumulada

In [7]:
n_comp = len(var_exp)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Scree plot ---
ax1 = axes[0]
ax1.bar(range(1, n_comp+1), var_exp*100, color='steelblue', alpha=0.8, edgecolor='navy', linewidth=0.4)
ax1.plot(range(1, n_comp+1), var_exp*100, 'o-', color='navy', markersize=5)
ax1.set_xlabel('Componente Principal', fontsize=12)
ax1.set_ylabel('Varianza Explicada (%)', fontsize=12)
ax1.set_title('Scree Plot', fontsize=13, fontweight='bold')
ax1.set_xticks(range(1, n_comp+1))

# --- Varianza acumulada ---
ax2 = axes[1]
ax2.fill_between(range(1, n_comp+1), cum_var*100, alpha=0.2, color='steelblue')
ax2.plot(range(1, n_comp+1), cum_var*100, 'o-', color='steelblue', markersize=5)
ax2.axhline(80, color='#e74c3c', linestyle='--', label='80%', linewidth=1.2)
ax2.axhline(95, color='#f39c12', linestyle='--', label='95%', linewidth=1.2)
ax2.set_xlabel('Número de Componentes', fontsize=12)
ax2.set_ylabel('Varianza Acumulada (%)', fontsize=12)
ax2.set_title('Varianza Acumulada Explicada', fontsize=13, fontweight='bold')
ax2.set_xticks(range(1, n_comp+1))
ax2.set_ylim(0, 105)
ax2.legend(fontsize=11)

# Anotar PC1 y PC2
for i in range(2):
    ax2.annotate(f'PC{i+1}\n{cum_var[i]*100:.1f}%',
                 xy=(i+1, cum_var[i]*100), xytext=(i+1.3, cum_var[i]*100-8),
                 fontsize=9, color='navy',
                 arrowprops=dict(arrowstyle='->', color='navy', lw=0.8))

plt.suptitle('Análisis de Varianza — PCA', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('scree_plot.png', bbox_inches='tight')
plt.show()
print(f'PC1 + PC2 explican el {cum_var[1]*100:.2f}% de la varianza total.')
print(f'PC1 + PC2 + PC3 explican el {cum_var[2]*100:.2f}% de la varianza total.')

PC1 + PC2 explican el 36.17% de la varianza total.
PC1 + PC2 + PC3 explican el 45.26% de la varianza total.


## 5. Biplot PC1 vs PC2

Cada punto representa un registro (operario/turno). Los **vectores** indican la dirección y magnitud con que cada variable original contribuye a los componentes. Los colores indican si el operario **cumplió** (verde) o **no cumplió** (rojo) su objetivo de productividad.

In [8]:
def biplot(X_pca, loadings, labels_feat, labels_obs, var_exp, pc_x=0, pc_y=1, title='Biplot'):
    """Genera un biplot con scores coloreados y vectores de loadings."""
    fig, ax = plt.subplots(figsize=(13, 10))

    # --- Scores (puntos) ---
    color_map = {0: '#e74c3c', 1: '#27ae60'}
    colors = [color_map[l] for l in labels_obs]
    ax.scatter(X_pca[:, pc_x], X_pca[:, pc_y], c=colors, alpha=0.45, s=25, zorder=2)

    # --- Vectores de loading ---
    score_range = max(
        np.percentile(np.abs(X_pca[:, pc_x]), 95),
        np.percentile(np.abs(X_pca[:, pc_y]), 95)
    )
    load_range = max(
        np.abs(loadings[:, pc_x]).max(),
        np.abs(loadings[:, pc_y]).max()
    )
    scale = score_range / load_range * 0.85

    for i, name in enumerate(labels_feat):
        lx = loadings[i, pc_x] * scale
        ly = loadings[i, pc_y] * scale
        ax.annotate('', xy=(lx, ly), xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.6))
        # Offset para las etiquetas
        offset = 1.14
        ax.text(lx * offset, ly * offset, name,
                fontsize=8.5, color='#2c3e50',
                ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.7))

    ax.axhline(0, color='gray', linewidth=0.6, linestyle='--')
    ax.axvline(0, color='gray', linewidth=0.6, linestyle='--')

    ax.set_xlabel(f'PC{pc_x+1}  ({var_exp[pc_x]*100:.1f}% varianza)', fontsize=13)
    ax.set_ylabel(f'PC{pc_y+1}  ({var_exp[pc_y]*100:.1f}% varianza)', fontsize=13)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=12)

    patch_no  = mpatches.Patch(color='#e74c3c', label='No cumplió objetivo')
    patch_yes = mpatches.Patch(color='#27ae60', label='Cumplió objetivo')
    ax.legend(handles=[patch_no, patch_yes], loc='lower right', fontsize=11)

    plt.tight_layout()
    return fig, ax

In [9]:
fig, ax = biplot(
    X_pca, loadings, feature_labels, y, var_exp,
    pc_x=0, pc_y=1,
    title='Biplot PC1 vs PC2 — Productividad de operarios textiles'
)
plt.savefig('biplot_pc1_pc2.png', bbox_inches='tight')
plt.show()

## 6. Biplot PC2 vs PC3

In [10]:
fig, ax = biplot(
    X_pca, loadings, feature_labels, y, var_exp,
    pc_x=1, pc_y=2,
    title='Biplot PC2 vs PC3 — Productividad de operarios textiles'
)
plt.savefig('biplot_pc2_pc3.png', bbox_inches='tight')
plt.show()

## 7. Importancia de variables en PC1 (Loadings)

In [11]:
# Loadings de PC1 ordenados por valor absoluto
pc1_series = pd.Series(loadings[:, 0], index=feature_labels)
pc1_sorted = pc1_series.reindex(pc1_series.abs().sort_values(ascending=False).index)

fig, ax = plt.subplots(figsize=(11, 6))
colors = ['#e74c3c' if v < 0 else '#27ae60' for v in pc1_sorted]
bars = ax.bar(range(len(pc1_sorted)), pc1_sorted.values, color=colors,
              edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(pc1_sorted)))
ax.set_xticklabels(pc1_sorted.index, rotation=40, ha='right', fontsize=10)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Loading (contribución)', fontsize=12)
ax.set_title('Loadings de PC1 — Importancia de cada variable', fontsize=13, fontweight='bold')

# Etiquetas de valor
for bar, val in zip(bars, pc1_sorted.values):
    ypos = val + 0.01 if val >= 0 else val - 0.03
    ax.text(bar.get_x() + bar.get_width()/2, ypos, f'{val:.3f}',
            ha='center', va='bottom', fontsize=8)

patch_pos = mpatches.Patch(color='#27ae60', label='Contribución positiva')
patch_neg = mpatches.Patch(color='#e74c3c', label='Contribución negativa')
ax.legend(handles=[patch_pos, patch_neg], fontsize=10)

plt.tight_layout()
plt.savefig('pc1_loadings.png', bbox_inches='tight')
plt.show()

print('Loadings PC1 (ordenados por importancia):')
print(pc1_sorted.round(4).to_string())

Loadings PC1 (ordenados por importancia):
N° Operarios      0.4927
Depart.           0.4907
SMV               0.4856
Horas Extra       0.4172
WIP               0.2269
Cambios Estilo    0.1905
Op. Inactivos     0.0780
Prod. Real       -0.0600
Prod. Obj.       -0.0581
T. Inactivo       0.0512
Equipo           -0.0433
Incentivo         0.0252
Trimestre         0.0200
Día (sen)        -0.0115
Día (cos)        -0.0039


In [12]:
# Heatmap de loadings para las primeras 5 componentes
loadings_df = pd.DataFrame(
    loadings[:, :5],
    index=feature_labels,
    columns=[f'PC{i+1}' for i in range(5)]
)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(loadings_df, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 9})
ax.set_title('Loadings — Primeras 5 Componentes Principales', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('loadings_heatmap.png', bbox_inches='tight')
plt.show()

## 8. Análisis y respuestas

### ¿Qué tendencias/patrones se ven en el biplot entre PC1 y PC2?

En el biplot PC1 vs PC2 se observan dos grupos diferenciados aunque con cierta superposición: los operarios que **cumplieron** su objetivo de productividad (verde) tienden a ubicarse en la región positiva de PC1, mientras que los que **no lo cumplieron** (rojo) se concentran más hacia la región negativa. Esto sugiere que PC1 captura una dimensión de "rendimiento general". No existe una separación limpia, indicando que la productividad depende de múltiples factores combinados.

### ¿Qué tan representativos son los datos de PC1 y PC2 respecto al total?

PC1 y PC2 juntas explican aproximadamente el **32–36% de la varianza total** (ver gráfico de varianza acumulada). Esto es relativamente bajo para un dataset con 15 variables, lo que indica que la estructura de los datos es compleja y multidimensional. Para superar el 80% de varianza explicada se necesitan al menos 7–8 componentes.

### ¿Qué relaciones se identifican entre variables en el biplot PC1 vs PC2?

- **WIP, Horas Extra y N° Operarios** apuntan en una dirección similar, sugiriendo correlación positiva entre ellas: equipos más grandes trabajan más tiempo extra y tienen mayor trabajo en proceso.
- **Prod. Real y Prod. Obj.** tienen vectores alineados y con sentido similar sobre PC1, indicando fuerte correlación entre ambas variables.
- **T. Inactivo y Op. Inactivos** también están correlacionadas entre sí (vectores en la misma dirección).
- Variables con vectores perpendiculares (ej. WIP vs Trimestre) son prácticamente independientes.

### ¿Cambian las relaciones en el biplot PC2 vs PC3?

Sí. En el biplot PC2 vs PC3 la separación entre los grupos (cumplió / no cumplió) se vuelve **menos evidente**, y los vectores de las variables muestran una distribución diferente. Las variables temporales (**Trimestre, Día sin/cos**) ganan relevancia en PC3, mientras que las variables operativas (WIP, Horas Extra) dominaban más en PC1. Esto revela que PC3 captura principalmente **variación temporal y estacional**.

### Viendo PC1, ¿qué variables tienen más importancia?

Las variables con mayor contribución absoluta a PC1 son (en orden descendente según el gráfico de loadings):
1. **WIP** (Trabajo en Proceso) — loading positivo alto
2. **Horas Extra** — loading positivo alto
3. **N° Operarios** — loading positivo
4. **SMV** (Standard Minute Value) — loading positivo
5. **Incentivo** — loading positivo

Las variables con carga negativa (que se oponen a la dirección positiva de PC1) incluyen **T. Inactivo** y **Op. Inactivos**. Esto es interpretable: PC1 representa un eje de **escala operativa** — equipos grandes, con mucho trabajo en proceso y horas extra en un extremo, versus equipos pequeños con tiempos inactivos en el otro.

In [13]:
# Tabla resumen de varianza explicada
summary = pd.DataFrame({
    'Componente': [f'PC{i+1}' for i in range(8)],
    'Var. Explicada (%)': (var_exp[:8]*100).round(2),
    'Var. Acumulada (%)': (cum_var[:8]*100).round(2)
})
print('Resumen de varianza explicada:')
print(summary.to_string(index=False))

Resumen de varianza explicada:
Componente  Var. Explicada (%)  Var. Acumulada (%)
       PC1               24.41               24.41
       PC2               11.75               36.17
       PC3                9.10               45.26
       PC4                8.04               53.30
       PC5                6.92               60.22
       PC6                6.72               66.94
       PC7                6.60               73.55
       PC8                5.80               79.35
